In [ ]:
# 미래를 예측하고 관계를 규명한다

# 회귀분석 : 하나 이상의 독립변수가 종속변수에 미치는 영향을 수학적으로 모델링하는 것
# 회귀분석의 목적 : 예측, 설명, 목적
# 주의 : 상관관계 != 인과관계 ()

In [ ]:
# 회귀모델 적합도 : 결정계수 R2
# 독립변수가 종속변수 변동의 영향을 미치는 정도를 설명
# R2 = 0.8 : 독립변수가 종속변수 변동의 80% 를 설명
# 0 <= R2 <= 1
# 1 에 가까울수록 설명력이 높음
# 모델 평가의 출발점일뿐, 유일한 기준은 아님 (R2, t-검정 등 많음)
# 실무기준 : R2 > 0.7 이면 좋은 모델, 0.5 이상이면 보통, 0.3 미만이면 개선이 필요한 모델


In [2]:
# 단순 선형 회귀 - 광고비와 매출의 관계
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

np.random.seed(42)

# 데이터 생성
n_samples = 100
advertising_spend = np.random.uniform(10, 100, n_samples)       # 광고비 (천 달러)
# 실제 관계 : Sales = 50 + 2.5 * Advertising + noise
true_slope = 2.5
true_intercept = 50
noise = np.random.normal(0, 10, n_samples)
sales = true_intercept + true_slope * advertising_spend + noise

# 데이터프레임 생성
regression_data = pd.DataFrame({
    'advertising': advertising_spend,
    'sales': sales
})

print("=== 광고비-매출 단순 선형 회귀 분석 ===")
print(f"샘플 수: {n_samples}")
print(f"실제 관계: Sales = {true_intercept} + {true_slope} * Advertising + noise")

# 1. statsmodels를 사용한 회귀분석
X = sm.add_constant(advertising_spend)      # 절편 추가
model = sm.OLS(sales, X)
results = model.fit()

print("\n=== 회귀분석 결과 요약 ===")
print(results.summary())

# 계수 추출
intercept = results.params[0]
slope = results.params[1]

print(f"\n=== 추정된 회귀식 ===")
print(f"Sales = {intercept:.2f} + {slope:.3f} * Advertising")
print(f"실제 회귀식: Sales = {true_intercept:.2f} + {true_slope:.3f} * Advertising")

# 2. 예측 및 잔차 분석
predictions = results.predict(X)
residuals = sales - predictions

# 성능 지표
mse = mean_squared_error(sales, predictions)
rmse = np.sqrt(mse)
mae = mean_absolute_error(sales, predictions)
r2 = r2_score(sales, predictions)

print(f"\n=== 모델 성능 지표 ===")
print(f"R2 : {r2:.4f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAE: {mae:.2f}")


=== 광고비-매출 단순 선형 회귀 분석 ===
샘플 수: 100
실제 관계: Sales = 50 + 2.5 * Advertising + noise

=== 회귀분석 결과 요약 ===
                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.981
Model:                            OLS   Adj. R-squared:                  0.981
Method:                 Least Squares   F-statistic:                     5171.
Date:                Thu, 12 Mar 2026   Prob (F-statistic):           1.30e-86
Time:                        17:02:23   Log-Likelihood:                -361.41
No. Observations:                 100   AIC:                             726.8
Df Residuals:                      98   BIC:                             732.0
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

np.random.seed(42)

In [ ]:
# 다중회귀 분석
# Y 값에 영향을 미치는 요인이 다양할 때
# 다중 선형 회귀 - 부동산 가격 예측

# 부동산 데이터 개수
n_houses = 500

# Feature 생성
size = np.random.normal(1500, 500, n_houses)        # 평방 피트
bedrooms = np.random.choice([1, 2, 3, 4, 5], n_houses, p=[0.1, 0.25, 0.35, 0.25, 0.05])     # 침실 개수
bathrooms = np.random.choice([1, 1.5, 2, 2.5, 3], n_houses, p=[0.2, 0.15, 0.35, 0.2, 0.1])      # 화장실 개수
age = np.random.uniform(0, 50, n_houses)            # 집 연식
location_score = np.random.uniform(1, 10, n_houses)     # 위치 점수
garage = np.random.choice([0, 1, 2, 3], n_houses, p=[0.2, 0.3, 0.4, 0.1])   # 차고 개수

# 실제 y 가격 계산 (천 달러)
# Price(y) = 50 + 0.1*size + 10*bedrooms + 15*bathrooms - 0.5*age + 8*location + 5*garage + noise
price = (
    50 
    + 0.1 * size
    + 10 * bedrooms
    + 15 * bathrooms
    - 0.5 * age
    + 8 * location_score
    + 5 * garage
    + np.random.normal(0, 20, n_houses)
)

# 데이터프레임 생성
df_house_data = pd.DataFrame({
    'size': size,
    'bedrooms': bedrooms,
    'bathrooms': bathrooms,
    'age': age,
    'location_score': location_score,
    'garage': garage,
    'price': price
})

print("=== 부동산 데이터 개요 ===")
print(df_house_data.info())
print("\n기술 통계:")
print(df_house_data.describe())

# == 상관관계 분석 == corr() 함수
correlation_matrix = df_house_data.corr()

# 다중 회귀 모델
X = df_house_data.drop('price', axis=1)
y = df_house_data['price']

# 훈련-테스트셋 분할
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# statsmodels 로 다중 회귀
X_train_sm = sm.add_constant(X_train)
X_test_sm = sm.add_constant(X_test)

model = sm.OLS(y_train, X_train_sm)
results = model.fit()           # 학습

print("\n=== 다중 회귀 분석 결과 ===")
print(results.summary())

# 예측
y_train_pred = results.predict(X_train_sm)
y_test_pred = results.predict(X_test_sm)

# 성능 평가
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))

print(f"\n=== 모델 성능 ===")
print(f"훈련 R2 : {train_r2:.4f}, RMSE: ${train_rmse:.2f}k")
print(f"테스트 R2 : {test_r2:.4f}, RMSE: ${test_rmse:.2f}k")

# coef 위주로 보면 됨


In [ ]:
# VIF 계산 (다중공선성 진단)
# 독립변수들 간에 높은 상관관계가 존재하는 현상

# - 회귀계수의 표준오차 증가
# - 회귀계수 해석의 어려움
# - 모델의 불안정성
# - 변수 선택의 어려움

from statsmodels.stats.outliers_influence import variance_inflation_factor

np.random.seed(42)

n_samples = 200

# 독립변수 설정
x1 = np.random.normal(0, 1, n_samples)
x2 = np.random.normal(0, 1, n_samples)
x3 = 2 * x1 + np.random.normal(0, 0.5, n_samples)           # x1과 높은 상관관계가 있는 식으로 제작
x4 = -0.5 * x2 + np.random.normal(0, 0.5, n_samples)        # x2와 중간정도 상관관계가 있는 식으로 제작
x5 = np.random.normal(0, 1, n_samples)                      # 독립적

# 종속 변수
y = 3 + 2*x1 - 1.5 * x2 + 0.5 * x3 + 1 * x4 + 2 * x5 + np.random.normal(0, 1, n_samples)

# 데이터프레임 생성
multicollinear_data = pd.DataFrame({
    'x1': x1,
    'x2': x2,
    'x3': x3,
    'x4': x4,
    'x5': x5,
    'y': y,
})

print("=== 다중공선성 진단 ===")

# 상관관계 행렬
corr_matrix = multicollinear_data.drop('y', axis=1).corr()
print("\n상관관계 행렬:")
print(corr_matrix.round(3))

# VIF 계산
X_vif = multicollinear_data.drop('y', axis=1)
vif_data = pd.DataFrame()
vif_data["Variable"] = X_vif.columns
vif_data["VIF"] = [variance_inflation_factor(X_vif.values, i) for i in range(len(X_vif.columns))]

print("\n=== VIF (Variance Inflation Factor) ===")
print(vif_data)
print("\nVIF 해석:")
print("  VIF < 5: 다중공선성 없음")
print("  5 <= VIF < 10: 중간 수준의 다중공선성")
print("  VIF >= 10: 심각한 다중공선성")

In [ ]:
# 종속변수가 이진형(0/1) 일때는 로지스틱 회귀 사용
# (구매/비구매, 이탈/유지, 성공/실패)
